# 대구교통공사 수송현황 머신러닝 분석

## 목표: 의사결정나무를 이용한 월별 수송인원 예측

## 0. 머신러닝과 의사결정나무

### 머신러닝이란?
- 데이터에서 패턴을 찾아 모델을 만드는 과정
- 함수 만들기와 유사 (입력 → 규칙 → 출력)
- 사람이 규칙을 정하지 않고 **컴퓨터가 스스로 규칙을 학습**

### 의사결정나무 (Decision Tree)
- 구조가 단순하고 **이해하기 쉬운 모델**
- 스무고개 놀이처럼 yes/no로 질문하며 분류
- 여러 복잡한 모델의 기초가 되는 알고리즘

### 모델 개발 순서
1. 데이터 읽어들이기
2. 데이터 확인
3. 데이터 전처리
4. 데이터 분할 (학습 80%, 검증 20%)
5. 알고리즘 선택 (의사결정나무)
6. 학습
7. 예측
8. 평가

## 1. 라이브러리 로드

In [15]:
# 01. 공통 코드 - 불필요한 경고 메시지 무시
import warnings
warnings.filterwarnings('ignore')

# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 한글 폰트 설정
import platform
from matplotlib import rc

if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    rc('font', family='Malgun Gothic')
else:
    rc('font', family='Malgun Gothic')

# 숫자 출력 조정
# 넘파이 부동소수점 출력 자리수 설정
np.set_printoptions(suppress=True, precision=4)

# 판다스 부동소수점 출력 자리수 설정
pd.options.display.float_format = '{:,.4f}'.format

# 데이터 프레임 모든 필드 출력
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# 그래프 글꼴 크기 설정
plt.rcParams['font.size'] = 12

# 난수 시드
random_seed = 123
np.random.seed(random_seed)

print("✓ 라이브러리 로드 및 설정 완료")

✓ 라이브러리 로드 및 설정 완료


## 2. 데이터 로드

In [16]:
from pathlib import Path

# 파일 경로 자동 찾기
notebook_dir = Path.cwd()
input_file = '대구교통공사_권종별월별수송현황_20260228.csv'

if (notebook_dir / 'input' / input_file).exists():
    input_path = notebook_dir / 'input' / input_file
elif (notebook_dir.parent / 'input' / input_file).exists():
    input_path = notebook_dir.parent / 'input' / input_file
else:
    raise FileNotFoundError(f"{input_file}을 찾을 수 없습니다.")

# 데이터 로드
df = pd.read_csv(str(input_path), encoding='cp949')

print(f"✓ 데이터 로드 완료")
print(f"데이터 크기: {df.shape}")
print(f"\n첫 5개 행:")
display(df.head())

✓ 데이터 로드 완료
데이터 크기: (340, 25)

첫 5개 행:


,년,월,선불카드(일반),선불카드(청소년),선불카드(어린이),선불카드(특수),후불카드(일반),후불카드(청소년),후불카드(어린이),아이조아선불(일반),아이조아선불(청소년),아이조아선불(어린이),아이조아후불(일반),무임카드(경로자),무임카드(장애인),무임카드(유공자),일반권,할인권,우대권(경로자),우대권(장애인),우대권(유공자),정액권,정기출입권,단체권,기타
0,1997,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,405719,38289,0,0,0,29715,0,501,17157
1,1997,12,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1659228,102474,0,0,0,394187,0,5416,87046
2,1998,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1358073,80955,104651,14968,6163,452714,0,28,67083
3,1998,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1206172,71691,107286,15345,6319,497108,0,32,56096
4,1998,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1226045,53002,130535,18670,7688,630144,0,95,55075


## 3. 데이터 확인

In [17]:
# 기본 정보
print("데이터 정보:")
print(df.info())

# 결측치 확인
print(f"\n결측치: {df.isnull().sum().sum()}개")

# 기본 통계
print(f"\n기본 통계:")
display(df.describe())

데이터 정보:
<class 'pandas.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 25 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   년            340 non-null    int64
 1   월            340 non-null    int64
 2   선불카드(일반)     340 non-null    int64
 3   선불카드(청소년)    340 non-null    int64
 4   선불카드(어린이)    340 non-null    int64
 5   선불카드(특수)     340 non-null    int64
 6   후불카드(일반)     340 non-null    int64
 7   후불카드(청소년)    340 non-null    int64
 8   후불카드(어린이)    340 non-null    int64
 9   아이조아선불(일반)   340 non-null    int64
 10  아이조아선불(청소년)  340 non-null    int64
 11  아이조아선불(어린이)  340 non-null    int64
 12  아이조아후불(일반)   340 non-null    int64
 13  무임카드(경로자)    340 non-null    int64
 14  무임카드(장애인)    340 non-null    int64
 15  무임카드(유공자)    340 non-null    int64
 16  일반권          340 non-null    int64
 17  할인권          340 non-null    int64
 18  우대권(경로자)     340 non-null    int64
 19  우대권(장애인)     340 non-null    int64
 20  우대권(유공자)     

,년,월,선불카드(일반),선불카드(청소년),선불카드(어린이),선불카드(특수),후불카드(일반),후불카드(청소년),후불카드(어린이),아이조아선불(일반),아이조아선불(청소년),아이조아선불(어린이),아이조아후불(일반),무임카드(경로자),무임카드(장애인),무임카드(유공자),일반권,할인권,우대권(경로자),우대권(장애인),우대권(유공자),정액권,정기출입권,단체권,기타
count,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000,340.0000
mean,"2,011.5000",6.5000,"2,154,538.2147","605,658.0735","91,414.2059",123.2000,"2,433,141.0559","22,229.7765",535.8265,"17,872.2765","26,875.3088","4,025.6147","9,810.7206","1,317,297.6088","166,748.3206","13,007.9265","924,863.4647","101,698.4588","529,566.4206","199,607.8971","18,250.3088","224,598.9265","12,351.9176","2,592.4882","123,695.6471"
std,8.1947,3.4798,"1,641,153.3657","472,381.4765","93,942.9398","1,374.4431","2,285,830.1886","49,954.5125","1,250.0970","34,054.3478","52,124.0131","9,107.3902","18,348.3212","1,162,947.9688","131,932.9493","10,851.1924","807,322.4997","39,796.2060","366,008.0165","91,884.8152","21,123.7662","450,810.9191","10,562.7375","3,179.3623","72,441.2339"
min,"1,997.0000",1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"68,106.0000","11,419.0000",0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"16,433.0000"
25%,"2,004.0000",3.0000,0.0000,0.0000,0.0000,0.0000,"115,103.5000",0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"315,274.0000","71,386.5000","329,912.2500","111,225.7500","6,121.2500",0.0000,0.0000,648.0000,"70,528.7500"
50%,"2,011.5000",6.5000,"1,949,819.0000","551,836.5000","87,797.0000",0.0000,"1,832,085.5000",0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"1,390,228.0000","231,008.0000","14,115.5000","618,550.5000","99,071.0000","399,983.0000","221,079.0000","12,240.0000",0.0000,"15,149.5000","1,507.5000","108,867.0000"
75%,"2,019.0000",10.0000,"3,737,897.2500","1,046,562.2500","116,326.2500",0.0000,"4,837,511.5000",0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"2,452,297.5000","279,294.7500","22,657.5000","1,429,954.7500","130,093.2500","493,396.5000","272,453.7500","16,763.2500","37,217.2500","20,635.2500","3,217.2500","167,323.2500"
max,"2,026.0000",12.0000,"5,105,325.0000","1,452,484.0000","447,397.0000","21,006.0000","6,674,180.0000","229,785.0000","6,226.0000","106,299.0000","174,517.0000","34,660.0000","53,683.0000","3,354,974.0000","366,222.0000","33,740.0000","3,172,769.0000","233,456.0000","1,737,574.0000","355,453.0000","88,334.0000","1,578,659.0000","33,703.0000","20,298.0000","637,857.0000"


## 4. 데이터 전처리

In [18]:
plt.tight_layout()

# 저장 경로 설정 (노트북 위치에 상관없이 올바른 output 폴더 찾기)
if (notebook_dir / 'output').exists():
    # 노트북이 2026-04-16 폴더에 있는 경우
    output_dir = notebook_dir / 'output'
elif (notebook_dir.parent / 'output').exists():
    # 노트북이 output 폴더에 있는 경우 (현재 상황)
    output_dir = notebook_dir.parent / 'output'
else:
    # 기본값
    output_dir = notebook_dir

output_path = output_dir / '머신러닝_분석결과.png'

# 경로가 없으면 생성
output_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(str(output_path), dpi=300, bbox_inches='tight')
print(f"✓ 그래프 저장: {output_path}")
plt.show()

✓ 그래프 저장: c:\Users\Administrator\컴비전학원\컴비젼5\2026-04-16\output\머신러닝_분석결과.png


<Figure size 640x480 with 0 Axes>

## 5. 데이터 분할

In [19]:
# 예측변수와 타겟변수 분리
X = df[['년', '월']].copy()
y = df['전체수송인원'].copy()

# 학습 데이터(80%) vs 검증 데이터(20%) 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("✓ 데이터 분할 완료")
print(f"\n학습 데이터: {X_train.shape[0]}개 (80%)")
print(f"검증 데이터: {X_test.shape[0]}개 (20%)")

KeyError: '전체수송인원'

## 6. 모델 구축 및 학습

In [ ]:
# 의사결정나무 모델 생성
model = DecisionTreeRegressor(max_depth=10, random_state=42)

# 학습
model.fit(X_train, y_train)

print("✓ 의사결정나무 모델 학습 완료")

## 7. 예측

In [ ]:
# 검증 데이터로 예측
y_pred = model.predict(X_test)

print("✓ 예측 완료")
print(f"\n예측값과 실제값 비교 (처음 10개):")
comparison = pd.DataFrame({
    '실제값': y_test.values[:10],
    '예측값': y_pred[:10],
    '오차': (y_test.values[:10] - y_pred[:10])
})
display(comparison)

## 8. 모델 평가

In [ ]:
# 평가 지표 계산
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("="*60)
print("의사결정나무 모델 평가")
print("="*60)
print(f"\n정확도 (R²):        {r2:.4f} ({r2*100:.2f}%)")
print(f"평균절대오차 (MAE):  {mae:,.0f}명")
print(f"제곱근평균제곱오차: {rmse:,.0f}명")

print(f"\n해석:")
print(f"- R² = {r2:.4f}: 모델이 데이터의 {r2*100:.2f}%를 설명")
print(f"- 평균 예측 오차: ±{mae:,.0f}명")

## 9. 결과 시각화

In [ ]:
# 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('대구교통공사 수송현황 머신러닝 분석', fontsize=16, fontweight='bold')

# 1. 실제값 vs 예측값
axes[0, 0].scatter(y_test, y_pred, alpha=0.6, s=80, color='steelblue')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
                'r--', lw=2, label='완벽한 예측')
axes[0, 0].set_xlabel('실제값 (명)', fontweight='bold')
axes[0, 0].set_ylabel('예측값 (명)', fontweight='bold')
axes[0, 0].set_title('실제값 vs 예측값', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. 시계열 - 월별 수송인원
df_sorted = df.sort_values('날짜')
axes[0, 1].plot(df_sorted['날짜'], df_sorted['전체수송인원'], 
               linewidth=2, color='darkgreen')
axes[0, 1].fill_between(df_sorted['날짜'], df_sorted['전체수송인원'], 
                       alpha=0.2, color='green')
axes[0, 1].set_xlabel('날짜', fontweight='bold')
axes[0, 1].set_ylabel('수송인원 (명)', fontweight='bold')
axes[0, 1].set_title('월별 수송인원 변화', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(True, alpha=0.3)

# 3. 오차 분포
residuals = y_test - y_pred
axes[1, 0].hist(residuals, bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1, 0].set_xlabel('오차 (명)', fontweight='bold')
axes[1, 0].set_ylabel('빈도', fontweight='bold')
axes[1, 0].set_title('예측 오차 분포', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. 평가 지표
metrics = ['R²', 'MAE\n(백만명)', 'RMSE\n(백만명)']
values = [r2, mae/1000000, rmse/1000000]
colors_bar = ['#2ecc71', '#3498db', '#e74c3c']

axes[1, 1].bar(metrics, values, color=colors_bar, edgecolor='black', linewidth=1.5, alpha=0.8)
axes[1, 1].set_ylabel('값', fontweight='bold')
axes[1, 1].set_title('모델 성능 지표', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

# 각 바에 값 표시
for i, (metric, value) in enumerate(zip(metrics, values)):
    axes[1, 1].text(i, value + 0.02, f'{value:.3f}', ha='center', fontweight='bold')

plt.tight_layout()

# 저장
output_dir = notebook_dir / 'output'
output_path = output_dir / '머신러닝_분석결과.png'
plt.savefig(str(output_path), dpi=300, bbox_inches='tight')
print(f"✓ 그래프 저장: {output_path}")
plt.show()

## 10. 미래 예측

In [ ]:
# 2026년 3월~12월 예측
future_data = pd.DataFrame({
    '년': [2026]*10,
    '월': list(range(3, 13))
})

future_pred = model.predict(future_data)

result = pd.DataFrame({
    '연도': future_data['년'],
    '월': future_data['월'],
    '예측 수송인원': future_pred.astype(int)
})

print("2026년 3월~12월 예측 수송인원:")
display(result)

## 11. 결론

In [ ]:
print("="*70)
print("분석 결론")
print("="*70)

print(f"""
1️⃣ 데이터
   - 기간: 1997년 11월 ~ 2026년 2월 (약 28년)
   - 데이터: {df.shape[0]}개월의 월별 수송인원 데이터
   - 결측치: 없음 ✓

2️⃣ 모델
   - 알고리즘: 의사결정나무 (Decision Tree)
   - 정확도(R²): {r2:.4f} ({r2*100:.2f}%)
   - 평균 오차: ±{mae:,.0f}명

3️⃣ 예측
   - 2026년 3월~12월 수송인원 예측 완료
   - 예측 범위: {result['예측 수송인원'].min():,.0f}명 ~ {result['예측 수송인원'].max():,.0f}명

4️⃣ 활용
   - 버스 기사 및 직원 배치 계획 수립
   - 유지보수 일정 관리
   - 경영 의사결정 지원
""")